# KGSS 성공 인식 분석

이 노트북은 KGSS 2009, 2014, 2021, 2023, 2025 자료를 사용해 한국 사회의 성공 요인 인식을 집계합니다.

원자료는 로컬의 `data/raw/kor_data_CUM0074_V2.sav`에만 두고, 이 노트북은 개인 단위 자료를 저장하지 않습니다. 저장되는 파일은 집계표와 그림뿐입니다.

## 1. 라이브러리 불러오기

데이터 처리에는 `pandas`와 `numpy`, SPSS 파일 읽기에는 `pyreadstat`, 그림 저장에는 `matplotlib.pyplot`, 경로 처리에는 `pathlib.Path`를 사용합니다.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt

## 2. 경로와 기본 설정

노트북은 `notebooks` 폴더에서 실행한다고 가정합니다. 원자료는 `../data/raw`에서 읽고, 집계표는 `../outputs/tables`, 그림은 `../outputs/figures`에 저장합니다.

In [ ]:
data_path = Path("../data/raw/kor_data_CUM0074_V2.sav")
table_dir = Path("../outputs/tables")
figure_dir = Path("../outputs/figures")

table_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

if not data_path.exists():
    raise FileNotFoundError(
        f"KGSS raw data file was not found at: {data_path}\n"
        "Place the local .sav file in data/raw and run this notebook again."
    )

## 3. KGSS 원자료 읽기

`pyreadstat.read_sav()`로 `.sav` 파일을 읽습니다. 분석에는 필요한 변수만 사용하며, 개인 단위 데이터는 따로 저장하지 않습니다.

In [ ]:
df, meta = pyreadstat.read_sav(data_path)

variables_needed = [
    "YEAR",
    "AGE",
    "FINALWT",
    "SUCDEFRT",
    "SUCDWLTH",
    "SUCDPAED",
    "SUCDKNOW",
    "KIDSOL06",
]

missing_variables = [variable for variable in variables_needed if variable not in df.columns]
if missing_variables:
    raise KeyError(f"Missing required variables: {missing_variables}")

analysis_df = df[variables_needed].copy()

print("Data shape:", df.shape)
print("Analysis variables:", analysis_df.columns.tolist())

## 4. 분석 변수와 응답 기준 정의

성공 인식 변수의 표시 이름을 정의합니다. 응답값 1, 2, 3은 중요하다고 보고, 4, 5는 중요하지 않다고 봅니다. 그 밖의 값은 분석에서 제외합니다.

In [ ]:
success_labels = {
    "SUCDEFRT": "열심히 일",
    "SUCDWLTH": "부유한 집안",
    "SUCDPAED": "부모 교육",
    "SUCDKNOW": "좋은 사람을 아는 것",
}

success_variables = list(success_labels.keys())
success_years = [2009, 2014, 2021, 2023, 2025]
important_values = [1, 2, 3]
valid_values = [1, 2, 3, 4, 5]

plot_colors = {
    "SUCDEFRT": "#2E86AB",
    "SUCDWLTH": "#A23B72",
    "SUCDPAED": "#F18F01",
    "SUCDKNOW": "#3B7A57",
}

## 5. 가중 중요 응답 비율 함수 만들기

`FINALWT`를 사용해 유효 응답자 중 '중요하다'고 본 응답의 가중 비율을 계산합니다. `valid_n`은 유효 응답값과 가중치가 모두 있는 사례 수입니다.

In [ ]:
def weighted_important_percentage(data, variable, weight_col="FINALWT"):
    """Return weighted important percentage and unweighted valid n for one variable."""
    valid_mask = data[variable].isin(valid_values) & data[weight_col].notna()
    valid_data = data.loc[valid_mask, [variable, weight_col]].copy()

    valid_n = len(valid_data)
    total_weight = valid_data[weight_col].sum()

    if valid_n == 0 or total_weight == 0:
        return np.nan, valid_n

    important_weight = valid_data.loc[
        valid_data[variable].isin(important_values), weight_col
    ].sum()
    weighted_pct = important_weight / total_weight * 100

    return weighted_pct, valid_n

## 6. 연도별 성공 요인 중요도 집계

2009, 2014, 2021, 2023, 2025년에 한해 네 가지 성공 요인의 가중 중요 응답 비율을 계산합니다. 결과는 집계표로 저장합니다.

In [ ]:
success_rows = []
success_df = analysis_df[analysis_df["YEAR"].isin(success_years)].copy()

for year in success_years:
    year_data = success_df[success_df["YEAR"] == year]

    for variable, label in success_labels.items():
        weighted_pct, valid_n = weighted_important_percentage(year_data, variable)
        success_rows.append(
            {
                "YEAR": year,
                "variable": variable,
                "label": label,
                "weighted_important_pct": weighted_pct,
                "valid_n": valid_n,
            }
        )

success_importance_by_year = pd.DataFrame(success_rows)
success_importance_by_year.to_csv(
    table_dir / "02_success_importance_by_year.csv",
    index=False,
)

success_importance_by_year

## 7. 노력 지수와 배경·관계 자본 지수 만들기

`열심히 일`의 가중 중요 응답 비율을 노력 지수로 사용합니다. `부유한 집안`, `부모 교육`, `좋은 사람을 아는 것`의 평균은 배경·관계 자본 지수로 해석합니다. 이 지수는 가족 배경뿐 아니라 좋은 사람을 아는 것처럼 개인 노력만으로 설명하기 어려운 관계 자본까지 포함합니다.

In [ ]:
success_wide = success_importance_by_year.pivot(
    index="YEAR",
    columns="variable",
    values="weighted_important_pct",
)

effort_vs_background_index = pd.DataFrame(
    {
        "YEAR": success_wide.index,
        "effort_index": success_wide["SUCDEFRT"],
        "background_index": success_wide[["SUCDWLTH", "SUCDPAED", "SUCDKNOW"]].mean(axis=1),
    }
).reset_index(drop=True)

effort_vs_background_index["effort_minus_background_gap"] = (
    effort_vs_background_index["effort_index"]
    - effort_vs_background_index["background_index"]
)

effort_vs_background_index.to_csv(
    table_dir / "02_effort_vs_background_index.csv",
    index=False,
)

effort_vs_background_index

## 8. 2025년 연령대별 성공 요인 중요도

2025년 응답자만 대상으로 20대, 30대, 40대, 50대, 60세 이상 집단을 만들고, 연령대별 성공 요인 중요도를 계산합니다.

In [ ]:
age_2025 = analysis_df[analysis_df["YEAR"] == 2025].copy()
age_2025["age_group"] = pd.cut(
    age_2025["AGE"],
    bins=[20, 30, 40, 50, 60, np.inf],
    labels=["20s", "30s", "40s", "50s", "60+"],
    right=False,
)

age_group_rows = []
age_group_order = ["20s", "30s", "40s", "50s", "60+"]

for age_group in age_group_order:
    group_data = age_2025[age_2025["age_group"] == age_group]

    for variable, label in success_labels.items():
        weighted_pct, valid_n = weighted_important_percentage(group_data, variable)
        age_group_rows.append(
            {
                "age_group": age_group,
                "variable": variable,
                "label": label,
                "weighted_important_pct": weighted_pct,
                "valid_n": valid_n,
            }
        )

success_importance_by_age_2025 = pd.DataFrame(age_group_rows)
success_importance_by_age_2025.to_csv(
    table_dir / "02_success_importance_by_age_2025.csv",
    index=False,
)

success_importance_by_age_2025

## 9. `KIDSOL06` 값 라벨과 연도별 원빈도 확인

`pyreadstat` 메타데이터에서 `KIDSOL06`의 값 라벨을 출력합니다. 이어서 연도별 응답값 원빈도를 집계해 저장합니다. 이 표 역시 개인 단위 자료가 아니라 연도별 집계 결과입니다.

In [ ]:
kidsol06_labels = meta.variable_value_labels.get("KIDSOL06", {})

print("KIDSOL06 value labels")
if kidsol06_labels:
    for value, label in kidsol06_labels.items():
        print(f"{value}: {label}")
else:
    print("No value labels found for KIDSOL06 in pyreadstat metadata.")

kidsol06_value_counts_by_year = (
    analysis_df.groupby(["YEAR", "KIDSOL06"], dropna=False)
    .size()
    .reset_index(name="raw_count")
    .sort_values(["YEAR", "KIDSOL06"], na_position="last")
)

kidsol06_value_counts_by_year.to_csv(
    table_dir / "02_kidsol06_value_counts_by_year.csv",
    index=False,
)

kidsol06_value_counts_by_year

## 10. 그림 저장을 위한 한글 글꼴 설정

그림에 한글이 깨지지 않도록 사용할 수 있는 한글 글꼴 후보를 설정합니다. 운영체제에 설치된 글꼴에 따라 표시 결과가 달라질 수 있습니다.

In [ ]:
from matplotlib import font_manager

font_candidates = [
    "AppleGothic",
    "Malgun Gothic",
    "NanumGothic",
    "NanumBarunGothic",
    "Noto Sans CJK KR",
    "Noto Sans KR",
    "Arial Unicode MS",
]

installed_font_names = {font.name for font in font_manager.fontManager.ttflist}
selected_font = next(
    (font_name for font_name in font_candidates if font_name in installed_font_names),
    None,
)

if selected_font is None:
    selected_font = "DejaVu Sans"
    print(
        "사용 가능한 한글 글꼴을 찾지 못했습니다. "
        "그림의 한글이 깨지면 AppleGothic, NanumGothic, Noto Sans KR 중 하나를 설치하세요."
    )
else:
    print(f"사용할 한글 글꼴: {selected_font}")

plt.rcParams["font.family"] = selected_font
plt.rcParams["axes.unicode_minus"] = False

## 11. 연도별 성공 요인 중요도 그림

각 성공 요인의 가중 중요 응답 비율이 시간에 따라 어떻게 변했는지 선 그래프로 확인하고 저장합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
label_y_offsets = {
    "SUCDEFRT": 0.45,
    "SUCDKNOW": -0.15,
    "SUCDWLTH": 0.0,
    "SUCDPAED": 0.0,
}

for variable, label in success_labels.items():
    plot_data = success_importance_by_year[
        success_importance_by_year["variable"] == variable
    ].sort_values("YEAR")
    ax.plot(
        plot_data["YEAR"],
        plot_data["weighted_important_pct"],
        marker="o",
        linewidth=2.4,
        color=plot_colors[variable],
    )
    last_row = plot_data.iloc[-1]
    ax.text(
        last_row["YEAR"] + 0.18,
        last_row["weighted_important_pct"] + label_y_offsets[variable],
        f"{label} {last_row['weighted_important_pct']:.1f}%",
        color=plot_colors[variable],
        va="center",
        fontsize=10,
    )

ax.set_title("성공 요인 인식 변화\n노력은 여전히 높고, 배경·관계 조건도 상승 (70-100% 확대)")
ax.set_xlabel("연도")
ax.set_ylabel("가중 중요 응답 비율 (%)")
ax.set_xticks(success_years)
ax.set_xlim(min(success_years) - 0.5, max(success_years) + 2.2)
ax.set_ylim(70, 100.5)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig(figure_dir / "02_success_importance_by_year.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. 노력 지수와 배경·관계 자본 지수 그림

노력 지수와 배경·관계 자본 지수를 비교하고, 두 지수 사이의 격차가 어떻게 줄었는지 따로 확인합니다. 이 그림은 배경이 노력보다 더 중요해졌다는 뜻이 아니라, 노력은 높게 유지되는 동안 배경·관계 조건의 중요도가 따라붙었다는 점을 보여줍니다.

In [ ]:
fig, (ax_index, ax_gap) = plt.subplots(
    1,
    2,
    figsize=(11, 4.8),
    gridspec_kw={"width_ratios": [1.5, 1]},
)

ax_index.plot(
    effort_vs_background_index["YEAR"],
    effort_vs_background_index["effort_index"],
    marker="o",
    linewidth=2.4,
    label="노력 지수",
    color="#2E86AB",
)
ax_index.plot(
    effort_vs_background_index["YEAR"],
    effort_vs_background_index["background_index"],
    marker="o",
    linewidth=2.4,
    label="배경 지수",
    color="#A23B72",
)
ax_index.set_title("중요 응답 비율 (80-100% 확대)")
ax_index.set_xlabel("연도")
ax_index.set_ylabel("가중 중요 응답 비율 (%)")
ax_index.set_xticks(success_years)
ax_index.set_ylim(80, 100.5)
ax_index.grid(axis="y", alpha=0.3)
ax_index.legend(frameon=False, loc="lower right")
ax_index.spines[["top", "right"]].set_visible(False)

ax_gap.bar(
    effort_vs_background_index["YEAR"].astype(str),
    effort_vs_background_index["effort_minus_background_gap"],
    color="#777777",
    alpha=0.75,
)
for _, row in effort_vs_background_index.iterrows():
    ax_gap.text(
        str(int(row["YEAR"])),
        row["effort_minus_background_gap"] + 0.35,
        f"{row['effort_minus_background_gap']:.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax_gap.set_title("노력 - 배경·관계 자본 격차")
ax_gap.set_xlabel("연도")
ax_gap.set_ylabel("격차 (%p)")
ax_gap.set_ylim(0, 15)
ax_gap.grid(axis="y", alpha=0.3)
ax_gap.spines[["top", "right"]].set_visible(False)

fig.suptitle("노력과 배경·관계 자본의 간격은 줄어들었다", y=1.04)
fig.tight_layout()
fig.savefig(figure_dir / "02_effort_vs_background_index.png", dpi=200, bbox_inches="tight")
plt.show()

## 13. 2025년 연령대별 성공 요인 중요도 그림

2025년 기준으로 연령대마다 어떤 성공 요인을 중요하게 보는지 비교합니다. 모든 항목의 응답 비율이 높기 때문에 80-100% 구간을 확대해 연령대별 차이가 보이도록 합니다.

In [ ]:
age_plot_data = success_importance_by_age_2025.pivot(
    index="age_group",
    columns="variable",
    values="weighted_important_pct",
).loc[age_group_order]

fig, ax = plt.subplots(figsize=(10, 5.5))
y_positions = np.arange(len(age_plot_data.index))

for variable in success_variables:
    ax.plot(
        age_plot_data[variable],
        y_positions,
        marker="o",
        linewidth=2.2,
        markersize=7,
        label=success_labels[variable],
        color=plot_colors[variable],
    )

ax.set_title("2025년 연령대별 성공 요인 중요도\n차이는 배경 요인에서 더 드러남 (80-100% 확대)")
ax.set_xlabel("가중 중요 응답 비율 (%)")
ax.set_ylabel("연령대")
ax.set_yticks(y_positions)
ax.set_yticklabels(age_plot_data.index)
ax.invert_yaxis()
ax.set_xlim(80, 101)
ax.grid(axis="x", alpha=0.3)
ax.legend(
    title="성공 요인",
    ncols=4,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.13),
    frameon=False,
)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig(figure_dir / "02_success_importance_by_age_2025.png", dpi=200, bbox_inches="tight")
plt.show()

## 14. 저장 결과 확인

마지막으로 저장된 집계표와 그림 파일 경로를 출력합니다. 이 노트북에서 저장한 결과물은 모두 집계 결과이며, 개인 단위 자료는 저장하지 않았습니다.

In [ ]:
saved_outputs = [
    table_dir / "02_success_importance_by_year.csv",
    table_dir / "02_effort_vs_background_index.csv",
    table_dir / "02_success_importance_by_age_2025.csv",
    table_dir / "02_kidsol06_value_counts_by_year.csv",
    figure_dir / "02_success_importance_by_year.png",
    figure_dir / "02_effort_vs_background_index.png",
    figure_dir / "02_success_importance_by_age_2025.png",
]

for output_path in saved_outputs:
    print(output_path)